# 第七课：配置、环境变量与日志

这一课假设你从未接触过环境变量和日志。

学习规则：

1. 一次只运行一个代码单元格；
2. 先观察输出，再阅读下面的解释；
3. 暂时不要直接完成整个`app_settings.py`；
4. 遇到不理解的输出就停下来提问。

本课最终要理解下面的流程：

```text
.env文件 → load_dotenv() → 环境变量 → os.getenv() → 程序配置
                                                       ↓
                                                    logging
```

## 第一部分：什么是配置？

程序运行时经常需要一些可以变化的数据：

- 应用名称；
- 日志详细程度；
- 数据库地址；
- 大模型API密钥。

这些数据称为**配置**。

下面是把配置直接写在Python代码中的例子。

In [1]:
app_name = "我的AI助手"
log_level = "INFO"

print(app_name)
print(log_level)

我的AI助手
INFO


这种代码可以运行，但如果把真实API密钥写进去并上传GitHub，其他人就能看到密钥。

此外，不同电脑和服务器可能需要不同配置。如果配置写死，每换一个环境都要修改代码。

因此我们希望：

```text
代码保持不变
配置从代码外面提供
```

## 第二部分：第一次读取环境变量

环境变量可以理解为程序外部提供的一组“名字和值”。Python使用`os.getenv()`读取它们。

先运行下面的单元格。这个变量目前不存在。

In [2]:
import os

result = os.getenv("A_VALUE_THAT_DOES_NOT_EXIST")

print(result)
print(type(result))

None
<class 'NoneType'>


找不到环境变量时，`os.getenv()`返回`None`。

也可以提供默认值。请运行：

In [3]:
result = os.getenv(
    "A_VALUE_THAT_DOES_NOT_EXIST",
    "这是默认值",
)

print(result)

这是默认值


现在可以总结：

```python
os.getenv("变量名")
```

- 找到：返回对应的字符串；
- 找不到：返回`None`。

```python
os.getenv("变量名", "默认值")
```

- 找到：返回对应的字符串；
- 找不到：返回提供的默认值。

## 第三部分：临时设置一个环境变量

在Python中可以通过`os.environ`临时设置环境变量。

In [ ]:
os.environ["DEMO_APP_NAME"] = "我的AI助手"

app_name = os.getenv("DEMO_APP_NAME")
print(app_name)

注意两种操作：

```python
os.environ["DEMO_APP_NAME"] = "我的AI助手"  # 设置
os.getenv("DEMO_APP_NAME")                  # 读取
```

这种设置主要用于实验和测试。关闭当前Python进程后，不应该指望它永久存在。

### 小检查1

请先猜输出，再运行：

In [4]:
os.environ["COLOR"] = "blue"

print(os.getenv("COLOR"))
print(os.getenv("SIZE"))
print(os.getenv("SIZE", "medium"))

blue
None
medium


## 第四部分：`.env`文件是什么？

如果配置很多，每次手动设置非常麻烦。开发时常把配置写进`.env`文件：

```text
APP_NAME=我的AI助手
LOG_LEVEL=INFO
LLM_API_KEY=练习密钥
```

它不是Python文件，每一行只是：

```text
名字=值
```

本课目录中有`.env.example`，它是安全模板。请在PowerShell执行：

```powershell
Copy-Item .env.example .env
```

然后把`.env`里的`LLM_API_KEY`改成练习值：

```text
LLM_API_KEY=abcd12345678wxyz
```

不要使用真实密钥。

In [6]:
from pathlib import Path

env_path = Path(".env")

print("Notebook当前目录：", Path.cwd())
print(".env是否存在：", env_path.exists())

if not env_path.exists():
    print("请先复制.env.example并命名为.env")

Notebook当前目录： 01_Python工程基础/07_配置环境变量与日志
.env是否存在： True


## 第五部分：把`.env`加载进环境变量

Python默认不会自动读取`.env`。`python-dotenv`提供了`load_dotenv()`：

In [ ]:
from dotenv import load_dotenv

loaded = load_dotenv()

print("是否找到并加载.env：", loaded)
print("APP_NAME：", os.getenv("APP_NAME"))
print("LOG_LEVEL：", os.getenv("LOG_LEVEL"))
print("LLM_API_KEY：", os.getenv("LLM_API_KEY"))


是否找到并加载.env： True
APP_NAME： test-app
LOG_LEVEL： warning
LLM_API_KEY： abcd12345678wxyz


In [25]:
loaded = load_dotenv()
loaded


True

刚才发生的过程是：

```text
.env文件
   ↓ load_dotenv()
环境变量
   ↓ os.getenv()
Python变量
```

练习时可以打印练习密钥，真实项目中不能打印真实密钥。

## 第六部分：`.env`和`.env.example`的区别

### `.env`

- 保存当前电脑使用的真实配置；
- 可能包含密钥；
- 不上传GitHub。

### `.env.example`

- 告诉别人项目需要哪些配置；
- 只包含安全示例；
- 需要上传GitHub。

可以把它们理解为：

```text
.env          填写好的私人表格
.env.example  空白表格模板
```

## 第七部分：第一次使用日志

`print()`适合临时查看内容。正式程序通常使用`logging`记录运行事件。

请运行：

In [10]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

logger = logging.getLogger("lesson07")

logger.debug("这是一条DEBUG日志")
logger.info("应用正常启动")
logger.warning("这是一条警告")
logger.error("某个操作失败")

2026-07-25 19:42:40,879 | INFO | 应用正常启动
2026-07-25 19:42:40,880 | WARNING | 这是一条警告
2026-07-25 19:42:40,880 | ERROR | 某个操作失败


### `logging.getLogger("lesson07")`到底做了什么？

`logging`是整个日志系统，`Logger`对象可以理解为某个程序模块写日志时使用的“记录员”。

```python
logger = logging.getLogger("lesson07")
```

这行代码分成三部分：

```text
logging                 Python日志模块
getLogger(...)          获取一个Logger对象
"lesson07"              这个Logger的名字
```

得到`logger`以后，才能用它记录不同级别的消息：

```python
logger.info("程序启动")
logger.error("请求失败")
```

名字不是要显示的消息，而是消息来源的标签。实际项目通常写：

```python
logger = logging.getLogger(__name__)
```

`__name__`会使用当前模块名，使日志能够说明消息来自哪个Python文件。

In [17]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s | %(name)s | %(message)s",
    force=True,
)

first_logger = logging.getLogger("lesson07")
same_logger = logging.getLogger("lesson07")
other_logger = logging.getLogger("database")

print("第一个名字：", first_logger.name)
print("另一个名字：", other_logger.name)
print("相同名字是否得到同一个Logger：", first_logger is same_logger)

first_logger.info("应用启动")
other_logger.warning("数据库连接较慢")

INFO | lesson07 | 应用启动
WARNING | database | 数据库连接较慢


第一个名字： lesson07
另一个名字： database
相同名字是否得到同一个Logger： True


### `basicConfig()`和`getLogger()`不是一回事

```python
logging.basicConfig(...)
```

负责设置日志系统的基本规则，例如最低级别和显示格式。

```python
logger = logging.getLogger("lesson07")
```

负责取得一个用来发送日志消息的Logger对象。

可以类比为：

```text
basicConfig()   制定公司的记录规则
getLogger()     找到某个部门的记录员
logger.info()   让记录员写下一条记录
```

所以`configure_logging()`只需要配置规则。下面这行如果创建后没有使用，就没有意义：

```python
logger = logging.getLogger("lesson07")
```

真正需要记录日志的模块再创建自己的logger，例如文件底部的：

```python
logger = logging.getLogger(__name__)
logger.info("应用启动")
```

你应该没有看到`DEBUG`，因为当前最低级别设置为`INFO`。

级别从低到高：

```text
DEBUG → INFO → WARNING → ERROR → CRITICAL
```

最低级别设置为`INFO`时，`INFO`及以上会显示，`DEBUG`不会显示。

In [11]:
logging.basicConfig(
    level=logging.WARNING,
    format="%(levelname)s | %(message)s",
    force=True,
)

logger = logging.getLogger("lesson07-warning")

logger.debug("DEBUG")
logger.info("INFO")
logger.warning("WARNING")
logger.error("ERROR")
logger.critical("CRITICAL")

WARNING | WARNING
ERROR | ERROR
CRITICAL | CRITICAL


日志可能被保存到文件或发送到监控平台，所以不能记录完整密钥：

```python
# 危险
logger.info("API密钥：%s", api_key)

# 较安全
logger.info("API密钥已加载")
logger.info("API密钥：%s", masked_key)
```

本课作业会要求你生成脱敏密钥，例如：

```text
abcd12345678wxyz → abcd...wxyz
```

## 第八部分：把作业拆成三个小步骤

### 步骤1：完成`masked_api_key()`

打开`app_settings.py`，只完成第一个TODO。

需要的知识：

```python
len(text)       # 字符串长度
text[:4]        # 前4个字符
text[-4:]       # 后4个字符
"*" * len(text) # 生成同样数量的星号
```

完成后运行下面的测试。

In [23]:
import importlib
import app_settings

importlib.reload(app_settings)

short = app_settings.Settings("app", "INFO", "1234")
long = app_settings.Settings("app", "INFO", "abcd12345678wxyz")

assert short.masked_api_key() == "****"
assert long.masked_api_key() == "abcd...wxyz"

print("步骤1通过")

步骤1通过


### 步骤2：完成`load_settings()`

先处理三个变量：

```python
app_name = os.getenv("APP_NAME", "ai-learning-app")
log_level = os.getenv("LOG_LEVEL", "INFO").upper()
api_key = os.getenv("LLM_API_KEY")
```

然后按作业文档验证`log_level`和`api_key`，最后返回：

```python
Settings(app_name, log_level, api_key)
```

请自己完成具体判断。

In [31]:
importlib.reload(app_settings)

os.environ["APP_NAME"] = "test-app"
os.environ["LOG_LEVEL"] = "warning"
os.environ["LLM_API_KEY"] = "abcd12345678wxyz"

settings = app_settings.load_settings()

assert settings.app_name == "test-app"
assert settings.log_level == "WARNING"
assert settings.api_key == "abcd12345678wxyz"

print("步骤2通过")

步骤2通过


### 步骤3：完成`configure_logging()`

字符串`"INFO"`需要转换成`logging.INFO`。可以使用：

```python
level = getattr(logging, log_level)
```

然后调用：

```python
logging.basicConfig(
    level=level,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
```

`force=True`在Notebook中特别有用，因为Notebook可能已经配置过日志。

In [32]:
importlib.reload(app_settings)

app_settings.configure_logging("WARNING")
root_logger = logging.getLogger()

assert root_logger.level == logging.WARNING
print("步骤3通过")

步骤3通过


## 第九部分：最终测试

三个步骤分别通过后，再运行完整测试。

In [33]:
importlib.reload(app_settings)

os.environ["APP_NAME"] = "test-app"
os.environ["LOG_LEVEL"] = "warning"
os.environ["LLM_API_KEY"] = "abcd12345678wxyz"

settings = app_settings.load_settings()
assert settings.masked_api_key() == "abcd...wxyz"

os.environ["LOG_LEVEL"] = "UNKNOWN"
try:
    app_settings.load_settings()
except ValueError:
    pass
else:
    raise AssertionError("非法LOG_LEVEL应该抛出ValueError")

os.environ["LOG_LEVEL"] = "INFO"
os.environ["LLM_API_KEY"] = "   "
try:
    app_settings.load_settings()
except RuntimeError:
    pass
else:
    raise AssertionError("空API密钥应该抛出RuntimeError")

print("第七课最终测试通过")

第七课最终测试通过


## 理解题1预备：配置和程序逻辑

先看两个东西：

```python
log_level = "WARNING"
```

`"WARNING"`是**配置值**。开发电脑可以用`DEBUG`，服务器可以用`WARNING`，不需要改变程序处理步骤。

```python
if log_level not in VALID_LOG_LEVELS:
    raise ValueError("日志级别不合法")
```

这段`if`是**程序逻辑**，它规定程序如何检查和处理配置。

再看一个生活类比：

```text
配置：空调目标温度设为24度
逻辑：低于目标温度就停止制冷，高于目标温度就继续制冷
```

配置回答“这次运行使用什么值”，程序逻辑回答“拿到这些值以后怎样处理”。

回答第1题时，可以分别举一个配置例子和一个程序逻辑例子，不要直接照抄上面的总结。

## 第十部分：理解题

请用自己的话回答：

1. 配置和普通程序逻辑有什么区别？请各举一个例子。
- 配置就是设定一个标准的值，而程序逻辑是根据这个标准的值给出不同的反应
- 例如配置了log_level，程序逻辑就规定了log_level为不同值的时候程序的不同反应
2. `os.getenv("NAME")`找不到变量时返回什么？怎样提供默认值？
- 找不到的时候返回None
- 通过os.getenv("NAME", "默认值")
3. `.env`和`.env.example`分别保存什么？为什么只有后者应该上传？
- `.env`提供的是真正的环境变量
- `.env.example`提供的是环境变量的模板
4. `load_dotenv()`做了什么？
- 将.env文件中的环境变量加载到程序中
5. 日志级别设置为`WARNING`时，哪些级别会显示？
- WARNING 、 ERROR 、 CRITICAL
6. 为什么日志中不能记录完整API密钥？
- 因为API秘钥是需要保密的，如果记录在日志中就会暴露